# 에이전트 디버깅(Agent Debugging)

이 노트북은 agent 시스템에서 문제가 생겼을 때 어디부터 봐야 하는지 알려주는 실전형 디버깅 튜토리얼이다. 최종 답만 보고 판단하지 않고, trace, node별 입력/출력, failure table을 함께 보면서 "이상이 시작된 첫 단계"를 찾는 연습을 한다.

## 학습 목표
- trace를 표로 읽고, node·timestamp·latency·inputs·outputs의 의미를 이해한다.
- `display_node_inputs`, `display_node_outputs`를 활용해 특정 노드를 좁혀보는 방법을 익힌다.
- 답변이 이상할 때 retrieval, synthesis, verification 중 어디서부터 의심해야 하는지 판단할 수 있다.
- 반복 evaluation 결과와 개별 trace를 연결해 recurring failure를 보는 법을 배운다.


## 개념 설명

디버깅 노트북도 먼저 현재 커널을 확인한다. trace와 evaluation 아티팩트를 읽어오는 노트북은 환경이 달라지면 결과 파일 경로가 꼬이기 쉽기 때문에, 시작 단계의 환경 확인이 곧 디버깅 준비다.

- **목적**: 현재 Python 환경이 올바른지 먼저 기록한다.
- **핵심 로직**: 프로젝트 루트 조정보다 앞서 `sys.executable`을 출력해 현재 커널을 확인한다.
- **주요 파라미터/변수**:
  - `sys.executable`: 노트북이 실제로 사용하는 Python 경로이다.

디버깅은 항상 관측 가능성(observability)에서 시작한다. 환경 정보 역시 그 관측의 일부다.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(sys.executable)

## 구현 준비

이 setup 셀은 기존 workflow와 evaluation 코드를 그대로 재사용하면서, 디버깅 전용 helper만 추가로 불러온다. 핵심은 workflow 자체를 뜯어고치는 것이 아니라, 이미 남겨진 trace와 state를 더 잘 읽는 것이다.

- **목적**: trace 시각화와 failure extraction에 필요한 함수들을 준비한다.
- **핵심 로직**: `display_trace`, `display_node_inputs`, `display_node_outputs`를 import하고, `build_demo_index(persist=False)`로 retriever를 만든다.
- **주요 파라미터/변수**:
  - `retriever`: debugging 대상 workflow 실행에 사용할 검색기이다.
  - `display_trace`: trace JSON을 표로 바꿔주는 진입점이다. 내부적으로는 trace를 데이터프레임 형태로 정리하는 변환 과정을 거친다.
  - `display_node_inputs`, `display_node_outputs`: 특정 node만 좁혀서 보는 helper이다.

즉, 이 셀은 "문제를 고치는 코드"가 아니라 "문제를 보이게 만드는 도구"를 세팅한다.


In [ ]:
import pandas as pd


from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from src.evaluator import extract_failure_cases, run_evaluation_suite
from src.ingestion import build_demo_index
from src.trace_debug import display_node_inputs, display_node_outputs, display_trace
from src.workflow import run_workflow

pd.set_option('display.max_colwidth', 140)
retriever = build_demo_index(persist=False)


## 디버깅 시나리오 시작

디버깅을 배울 때는 성공 케이스 하나와 실패 또는 abstain 케이스 하나를 같이 보는 것이 좋다. 그래야 정상 경로와 비정상 경로가 어떻게 갈라지는지 비교할 수 있다. 여기서는 날짜 계산 질문과 문서 범위 밖 CEO 질문을 나란히 실행한다.

- **목적**: 정상적인 answered 경로와 abstained 경로를 한 번에 준비한다.
- **핵심 로직**: `run_workflow()`를 두 번 호출해 `happy_state`와 `abstain_state`를 만들고, 최종 상태와 trace step 수를 요약한다.
- **주요 파라미터/변수**:
  - `happy_state`: 문서와 tool로 해결 가능한 질문의 실행 결과이다.
  - `abstain_state`: 근거 부족으로 멈춰야 하는 질문의 실행 결과이다.
  - `happy_steps`, `abstain_steps`: 두 경로의 실행 길이를 비교하는 지표이다.

이 셀의 요약은 이후 디버깅의 기준선 역할을 한다. 정상 케이스와 이상 케이스를 같이 보면, 어디서부터 경로가 달라졌는지 더 쉽게 찾을 수 있다.


In [ ]:
happy_state = run_workflow('How many days are in the pilot window?', retriever=retriever)
abstain_state = run_workflow('Who is the current CEO of the company?', retriever=retriever)
{
    'happy_status': happy_state['final_status'],
    'abstain_status': abstain_state['final_status'],
    'happy_steps': len(happy_state['trace']),
    'abstain_steps': len(abstain_state['trace']),
}


## 전체 trace 읽기

디버깅의 첫 번째 도구는 전체 trace 테이블이다. `display_trace()`는 내부적으로 trace JSON 리스트를 사람이 읽기 쉬운 표로 바꿔준다. 흔히 `trace_to_debug_frame` 같은 변환 단계를 거쳐, 각 노드의 핵심 메타데이터를 열(column)로 정리한 뒤 출력한다고 이해하면 된다.

- **목적**: 개별 node가 어떤 순서와 결과로 실행됐는지 전체 그림을 본다.
- **핵심 로직**: `display_trace(happy_state['trace'], render=False)`가 trace를 화면 출력 대신 데이터프레임으로 반환한다.
- **주요 파라미터/변수**:
  - `node`: 실행된 workflow 단계 이름이다.
  - `timestamp`: trace가 기록된 시점이다.
  - `latency`: 해당 node가 실행되는 데 걸린 시간이다.
  - `inputs`, `outputs`: node가 받은 값과 만든 값을 요약한 열이다.

전체 trace를 읽을 때는 먼저 node 순서가 기대한 workflow와 맞는지 보고, 그다음 어떤 단계의 출력이 비어 있거나 이상한지 찾는다. 최종 답이 틀려도, 실제 문제는 앞쪽 retrieval이나 classification에서 시작된 경우가 많다.


In [ ]:
debug_trace_frame = display_trace(happy_state['trace'], render=False)
debug_trace_frame


## Trace 성능 분석(Trace Performance Analysis)

trace는 correctness뿐 아니라 성능 관점에서도 유용하다. 어떤 node가 예상보다 느린지 보면, 병목이 retrieval인지 tool인지 verification인지 감을 잡을 수 있다. 여기서는 전체 trace 중 `node`, `latency`, `inputs`, `outputs` 열만 뽑아 성능 중심으로 본다.

- **목적**: 디버깅을 기능 이상뿐 아니라 실행 시간 관점으로도 확장한다.
- **핵심 로직**: `display_trace(...)[['node', 'latency', 'inputs', 'outputs']]`로 성능 분석에 필요한 열만 추린다.
- **주요 파라미터/변수**:
  - `performance_frame`: node별 지연 시간 분석용 데이터프레임이다.
  - `latency`: 초 또는 밀리초 단위의 실행 시간 요약이다.

이 표에서는 절대값보다 상대값이 중요하다. normalization보다 retrieval이 느린 것은 자연스럽지만, 단순 fallback이 지나치게 느리다면 이상 신호일 수 있다.


In [ ]:
performance_frame = display_trace(happy_state['trace'], render=False)[['node', 'latency', 'inputs', 'outputs']]
performance_frame

latency가 높다고 해서 자동으로 나쁜 것은 아니다. 중요한 질문은 그 시간이 해당 작업에 비해 자연스러운지 여부다. 예를 들어 retrieval이 normalization보다 느린 것은 자연스럽지만, tool이나 verification이 과도하게 느리다면 입력 크기, 외부 의존성, 직렬화 비용을 의심해볼 수 있다.

또 하나 중요한 점은, 성능 분석만으로는 원인을 알 수 없다는 것이다. 특정 node가 느리거나 결과가 비어 보이면, 그다음에는 그 node의 입력과 출력을 따로 봐야 한다. 그래서 아래 두 셀처럼 node별 세부 뷰가 필요하다.


## 노드 입력 점검(Node Inputs)

어떤 노드가 이상해 보이면 먼저 입력부터 확인하는 것이 좋다. 출력이 이상한 이유가 그 노드 자체의 버그일 수도 있지만, 사실은 이전 단계가 이미 잘못된 입력을 넘겼기 때문일 수 있다.

- **목적**: 특정 node가 실제로 어떤 입력을 받았는지 확인한다.
- **핵심 로직**: `display_node_inputs(happy_state['trace'], 'retrieve_docs', render=False)`가 `retrieve_docs` node의 입력만 표로 보여준다.
- **주요 파라미터/변수**:
  - `'retrieve_docs'`: 살펴볼 대상 node 이름이다.
  - `render=False`: 노트북 렌더링 대신 데이터프레임을 반환해 추가 분석이 가능하게 한다.

실제 디버깅에서는 여기서 질문 문자열이 예상과 다르게 정규화됐는지, `top_k`가 지나치게 작거나 큰지, retriever에 전달된 상태가 비어 있지는 않은지를 본다.


In [ ]:
display_node_inputs(happy_state['trace'], 'retrieve_docs', render=False)


## 노드 출력 점검(Node Outputs)

입력이 정상이었다면 다음은 출력이다. node 출력은 그 단계가 실제로 무엇을 만들어냈는지 보여준다. retrieval 노드라면 어떤 문서를 찾았는지, verifier 노드라면 coverage가 얼마였는지, fallback 노드라면 왜 abstain했는지를 여기서 확인한다.

- **목적**: 특정 node가 만든 결과를 입력과 분리해 점검한다.
- **핵심 로직**: `display_node_outputs(happy_state['trace'], 'retrieve_docs', render=False)`가 `retrieve_docs` node의 출력만 표로 보여준다.
- **주요 파라미터/변수**:
  - `outputs`: node가 상태에 남긴 핵심 결과 요약이다.

노드별 input/output을 볼 수 있다는 것은, agent 디버깅을 "최종 답이 이상하다" 수준에서 끝내지 않고 "retrieval이 빈약해서 synthesis가 흔들렸다"처럼 원인 중심으로 바꿀 수 있다는 뜻이다.


In [ ]:
display_node_outputs(happy_state['trace'], 'retrieve_docs', render=False)


## 실험

이 셀은 성공 케이스와 abstain 케이스를 간단한 비교표로 정리한다. 답변 상태, coverage score, unsupported claim 수를 함께 보면, 왜 한 질문은 answered 되었고 다른 질문은 abstained 되었는지 더 명확히 읽을 수 있다.

- **목적**: 두 실행 경로를 핵심 지표 중심으로 비교한다.
- **핵심 로직**: `happy_state`와 `abstain_state`에서 최종 상태와 verification 관련 값을 뽑아 `comparison` 데이터프레임으로 만든다.
- **주요 파라미터/변수**:
  - `coverage_score`: 답변이 근거 문서에 얼마나 덮여 있는지 보여준다.
  - `unsupported_claims`: 문서에서 지지되지 않은 주장 수이다.

이 표를 볼 때는 answered/abstained만 보지 말고, 그 뒤에 있는 verifier 신호를 함께 보자. 디버깅은 상태 라벨보다 근거가 되는 중간 신호를 읽는 작업이다.


In [ ]:
comparison = pd.DataFrame(
    [
        {
            'query': happy_state['user_query'],
            'final_status': happy_state['final_status'],
            'coverage_score': happy_state['verification_result'].coverage_score,
            'unsupported_claims': len(happy_state['verification_result'].unsupported_claims),
        },
        {
            'query': abstain_state['user_query'],
            'final_status': abstain_state['final_status'],
            'coverage_score': abstain_state['verification_result'].coverage_score,
            'unsupported_claims': len(abstain_state['verification_result'].unsupported_claims),
        },
    ]
)
comparison


디버깅은 반복 evaluation과 함께 볼 때 더 강해진다. 한두 개 trace만 보면 우연인지 패턴인지 알기 어렵기 때문이다. 이 셀은 작은 evaluation run을 돌린 뒤 failure row만 추려서, dataset 수준에서 어떤 문제가 반복되는지 보여준다.

- **목적**: 개별 디버깅을 반복 실패 패턴 분석으로 확장한다.
- **핵심 로직**: `run_evaluation_suite(repeats=1, persist_outputs=True)`로 작은 평가를 실행하고, `extract_failure_cases(results)`로 실패만 뽑아 본다.
- **주요 파라미터/변수**:
  - `results`: 전체 평가 결과 데이터프레임이다.
  - `failures`: 실패 케이스만 추린 표이다.
  - `failure_type`, `predicted_status`: 반복적으로 관찰되는 문제 유형과 시스템 반응을 보여준다.

이 표는 "답변이 이상할 때 어디부터 봐야 하는가"에 대한 힌트를 준다. 같은 failure type이 계속 반복되면 개별 질문이 아니라 시스템 구성요소 하나를 고쳐야 할 가능성이 크다.


In [ ]:
results, summary = run_evaluation_suite(repeats=1, persist_outputs=True)
failures = extract_failure_cases(results)
failures[['system', 'question_id', 'question', 'failure_type', 'predicted_status']].head(10)


## 결과 해석

이 마지막 분석 셀은 디버깅에서 자주 쓰는 관측 뷰를 한 표로 정리한다. 전체 trace, node inputs, node outputs, failure table은 서로 다른 해상도(resolution)의 관찰 도구라고 생각하면 된다.

- **목적**: 디버깅 도구를 상황별로 어떻게 선택할지 정리한다.
- **핵심 로직**: `analysis_frame`에 각 debugging view와 대표 사용처를 표로 저장한다.
- **주요 파라미터/변수**:
  - `full_trace`: 전체 흐름을 볼 때 쓴다.
  - `node_inputs`: 이전 단계에서 잘못된 값이 들어왔는지 확인할 때 쓴다.
  - `node_outputs`: 해당 노드 자체의 산출물이 이상한지 볼 때 쓴다.
  - `failure_table`: 반복적인 시스템 문제를 찾을 때 쓴다.

실제 디버깅 순서는 보통 이렇다.
1. 최종 상태와 전체 trace를 본다.
2. 이상해 보이는 node를 고른다.
3. 그 node의 inputs/outputs를 분리해 본다.
4. evaluation failure table에서 같은 패턴이 반복되는지 확인한다.

이 흐름을 익히면 agent 디버깅이 감이 아니라 절차가 된다.


In [ ]:
analysis_frame = pd.DataFrame(
    [
        {'debugging_view': 'full_trace', 'use_case': 'understand the whole execution path'},
        {'debugging_view': 'node_inputs', 'use_case': 'inspect what information reached a node'},
        {'debugging_view': 'node_outputs', 'use_case': 'inspect what the node actually produced'},
        {'debugging_view': 'failure_table', 'use_case': 'spot recurring issues across many runs'},
    ]
)
analysis_frame


## 핵심 정리

이 노트북을 통해 agent 디버깅은 최종 답을 눈으로 읽는 작업이 아니라, trace와 state를 따라가며 이상이 시작된 첫 지점을 찾는 과정이라는 점을 확인했다. 전체 trace는 거시적 흐름을 보여주고, node input/output 뷰는 원인 추적을 더 정밀하게 만들어준다.

또한 반복 evaluation과 failure extraction을 함께 보면, 개별 버그처럼 보이던 현상이 사실은 구조적 패턴일 수 있다는 점도 확인할 수 있다. 즉, 좋은 디버깅은 개별 사례 분석과 집계 수준 분석을 오가야 한다.

💡 면접 포인트: "agent 시스템은 black box처럼 보이기 쉽지만, trace·node I/O·failure table을 함께 보면 어느 단계에서 품질이 무너졌는지 재구성할 수 있다"고 설명하면 좋다.
